# 🐍 Part 2: Database Enrichment & Data Wrangling via Python

## Refactoring and Enriching the `Customers` Table (`04_update_customers.ipynb`)

## 🎯 Context & Objectives

The goal of this stage is to perform strategic data enrichment on the `customers` table within the `insight_places` database using Python (`04_update_customers.ipynb`). By standardizing personal identifiers to English (`full_name`), updating contact formats (`email`, `phone_number`), enforcing global platform preferences (`preferred_currency`, `preferred_language`), adding socio-demographic features (`birth_date`, `gender`, `travel_profile`), and establishing billing address attributes, I will turn this table into a robust customer dimension for advanced cohort analysis, RFM segmentation, and BI dashboards.

This notebook represents the fourth module of a sequential ETL execution, preserving complete relational integrity with the `bookings` table.

## 🛠️ Environment Setup & Dependencies

To execute this ETL pipeline securely and efficiently, the following Python libraries are required:

* **`sqlalchemy` & `pymysql`:** Database abstraction layer and MySQL driver for executing DDL/DML operations.
* **`python-dotenv`:** Security protocol to parse database credentials from the existing `.env` environment file without exposing sensitive keys.
* **`pandas`:** Data manipulation, aggregation, and structural validation.
* **`random` & `numpy`:** Native and mathematical modules used to generate realistic pricing distributions and conditional business logic.
* **`faker`:** Synthetic data generation library used to produce realistic English names, addresses, and identity metadata (`en_US` locale).

## 📌 Refining the `Customers` Table

This module focuses on upgrading customer profiles, socio-demographic metadata, and platform preferences:

1. **Schema Modernization & Identity Enrichment:** Updating existing columns and adding structural attributes (`gender`, `birth_date`, `document_type`, `document_number`, `is_identity_verified`).
2. **Name & Contact Standardization (`full_name`, `email`, `phone_number`):** Generating realistic English names using `Faker`, setting clean emails with real domains, and formatting phone numbers.
3. **Platform Preferences & Behavior (`travel_profile`, `preferred_payment_method`):** Mapping user behaviors (`Leisure`, `Business`, `Digital Nomad`) and setting global currency and language configurations.
4. **Billing Address & Spatial Metadata (`city`, `state`, `country`):** Structuring localized billing addresses to support spatial origin analytics.
5. **Temporal Integrity Enforcer (`joined_at`, `last_login_at`):** Establishing registration and activity timestamps that precede customer booking activities.
6. **Governance & Privacy (`marketing_opt_in`):** Setting compliance flags for promotional communications (GDPR/LGPD compliance simulation).

### 1) Installing and importing the libraries

In this step, I will load the libraries that will be used throughout the project.

In [1]:
!pip install sqlalchemy
!pip install pymysql
!pip install faker

import pandas as pd
import random
import datetime
import pymysql
from faker import Faker
from sqlalchemy import create_engine, text

fake = Faker('en_US')

#### 1.1) Installing the data protection library

I will install the python-dotenv library to ensure the security of the MySQL password.

In [2]:
!pip install python-dotenv

#### 1.2) Environment Setup and Security Validation

In [3]:
# Installing and Importing libraries
import os
from dotenv import load_dotenv

# Loading variables from the .env file
load_dotenv()

# Assigning credentials to variables using os.getenv
user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST', 'localhost')
port = os.getenv('DB_PORT', '3306')
database = os.getenv('DB_NAME', 'insight_places')

# Security Check: Validating if environment variables are present
if all([user, password, host, database]):
    print("🔒 Environment variables loaded successfully.")
    print(" System ready for database connection.") 
else:
    print("❌ Error: One or more credentials are missing from the .env file.")

🔒 Environment variables loaded successfully.
 System ready for database connection.


#### 1.3) Database Engine Initialization

Using the validated credentials, I will use SQLAlchemy to create the engine. This object will be the communication bridge between Python and MySQL, allowing for efficient data transfer.

In [4]:
# Creating the database connection engine using validated credentials

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")
print("🚀 Engine connection is ready!")

🚀 Engine connection is ready!


## 2) Execution: Refining and Enriching the `Customers` Table

This section executes the step-by-step transformation pipeline to modernize customer identity, apply behavioral preferences, structure billing metadata, and enforce temporal sequence rules.

### 2.1 Schema Modernization (DDL Adjustments)

In this step, I modify the `customers` table structure by renaming or adding attributes (`full_name`, `gender`, `birth_date`, `phone_number`, `document_type`, `document_number`, `is_identity_verified`, `street`, `city`, `state`, `zip_code`, `country`, `preferred_language`, `preferred_currency`, `preferred_payment_method`, `travel_profile`, `joined_at`, `last_login_at`, `marketing_opt_in`) to support enriched analytical queries and segmentation models.

In [5]:
def update_customers_schema():
    with engine.begin() as conn:
        print("Checking existing columns in 'customers' table...")
        
        # Inspection of current columns
        columns_query = text("""
            SELECT COLUMN_NAME 
            FROM INFORMATION_SCHEMA.COLUMNS 
            WHERE TABLE_SCHEMA = DATABASE() AND TABLE_NAME = 'customers'
        """)
        existing_columns = [row[0] for row in conn.execute(columns_query).fetchall()]
        
        # Dictionary of required new columns and their SQL definitions
        columns_to_add = {
            'full_name': 'VARCHAR(255)',
            'gender': 'VARCHAR(20)',
            'birth_date': 'DATE',
            'phone_number': 'VARCHAR(30)',
            'document_type': 'VARCHAR(20)',
            'document_number': 'VARCHAR(50)',
            'is_identity_verified': 'TINYINT DEFAULT 0',
            'street': 'VARCHAR(255)',
            'city': 'VARCHAR(100)',
            'state': 'VARCHAR(100)',
            'zip_code': 'VARCHAR(20)',
            'country': 'VARCHAR(100)',
            'preferred_language': 'VARCHAR(10) DEFAULT "en-US"',
            'preferred_currency': 'VARCHAR(10) DEFAULT "USD"',
            'preferred_payment_method': 'VARCHAR(50)',
            'travel_profile': 'VARCHAR(50)',
            'joined_at': 'DATETIME',
            'last_login_at': 'DATETIME',
            'marketing_opt_in': 'TINYINT DEFAULT 1'
        }
        
        # 1. Rename legacy 'name' column to 'full_name' if applicable
        if 'name' in existing_columns and 'full_name' not in existing_columns:
            conn.execute(text("ALTER TABLE customers CHANGE COLUMN name full_name VARCHAR(255);"))
            print("  `name` column successfully renamed to `full_name`.")
            
        # 2. Add missing columns
        for col_name, col_type in columns_to_add.items():
            if col_name not in existing_columns and col_name != 'full_name':
                conn.execute(text(f"ALTER TABLE customers ADD COLUMN {col_name} {col_type};"))
                print(f"  `{col_name}` column successfully added.")

        print("✅ Customer schema modernization completed successfully!")

update_customers_schema()

Checking existing columns in 'customers' table...
  `gender` column successfully added.
  `birth_date` column successfully added.
  `phone_number` column successfully added.
  `document_type` column successfully added.
  `document_number` column successfully added.
  `is_identity_verified` column successfully added.
  `street` column successfully added.
  `city` column successfully added.
  `state` column successfully added.
  `zip_code` column successfully added.
  `country` column successfully added.
  `preferred_language` column successfully added.
  `preferred_currency` column successfully added.
  `preferred_payment_method` column successfully added.
  `travel_profile` column successfully added.
  `joined_at` column successfully added.
  `last_login_at` column successfully added.
  `marketing_opt_in` column successfully added.
✅ Customer schema modernization completed successfully!


### 2.2 Name, Identity & Contact Enrichment (`full_name`, `gender`, `birth_date`, `email`, `phone_number`, `document_type`, `document_number`, `is_identity_verified`)

Standardizing customer profile identities using `Faker` to generate English full names, assigning consistent gender classifications, calculating realistic adult birth dates (ages 18 to 70), establishing valid contact details, and assigning identity verification status.

In [9]:
def update_customer_identity_and_contact():
    with engine.begin() as conn:
        print("Checking schema consistency and fetching customer records...")
        
        # 1. Inspect existing columns
        columns_query = text("""
            SELECT COLUMN_NAME 
            FROM INFORMATION_SCHEMA.COLUMNS 
            WHERE TABLE_SCHEMA = DATABASE() AND TABLE_NAME = 'customers'
        """)
        existing_columns = [row[0] for row in conn.execute(columns_query).fetchall()]
        
        # Ensure customer_name exists and drop any temporary full_name column
        if 'full_name' in existing_columns and 'customer_name' not in existing_columns:
            conn.execute(text("ALTER TABLE customers CHANGE COLUMN full_name customer_name VARCHAR(255);"))
        elif 'full_name' in existing_columns and 'customer_name' in existing_columns:
            conn.execute(text("ALTER TABLE customers DROP COLUMN full_name;"))
            
        if 'customer_name' not in existing_columns and 'name' in existing_columns:
            conn.execute(text("ALTER TABLE customers CHANGE COLUMN name customer_name VARCHAR(255);"))
            
        # Ensure email column exists and has proper length
        if 'contact' in existing_columns and 'email' not in existing_columns:
            conn.execute(text("ALTER TABLE customers CHANGE COLUMN contact email VARCHAR(255);"))
        elif 'email' not in existing_columns:
            conn.execute(text("ALTER TABLE customers ADD COLUMN email VARCHAR(255);"))
            
        # Fetch customer IDs
        customers = conn.execute(text("SELECT customer_id FROM customers")).fetchall()
        
        genders = ['Female', 'Male', 'Non-Binary']
        gender_weights = [0.49, 0.48, 0.03]
        
        doc_types = ['Passport', 'SSN', 'Driver License']
        doc_weights = [0.40, 0.45, 0.15]
        
        domains = ['gmail.com', 'yahoo.com', 'outlook.com', 'icloud.com']
        updates = []
        
        today = datetime.date(2026, 1, 1)
        
        for (customer_id,) in customers:
            selected_gender = random.choices(genders, weights=gender_weights)[0]
            
            if selected_gender == 'Female':
                full_name = fake.name_female()
            elif selected_gender == 'Male':
                full_name = fake.name_male()
            else:
                full_name = fake.name()
                
            clean_handle = full_name.lower().replace(" ", ".").replace("'", "")
            domain = random.choice(domains)
            email = f"{clean_handle}@{domain}"
            
            phone = f"+1 (555) {random.randint(100, 999)}-{random.randint(1000, 9999)}"
            
            age_in_days = random.randint(18 * 365, 70 * 365)
            birth_date = (today - datetime.timedelta(days=age_in_days)).strftime('%Y-%m-%d')
            
            doc_type = random.choices(doc_types, weights=doc_weights)[0]
            doc_number = fake.bothify(text='???-##-####').upper()
            is_verified = 1 if random.random() < 0.82 else 0
            
            updates.append({
                "customer_id": customer_id,
                "customer_name": full_name,
                "gender": selected_gender,
                "birth_date": birth_date,
                "email": email,
                "phone_number": phone,
                "document_type": doc_type,
                "document_number": doc_number,
                "is_identity_verified": is_verified
            })
            
        update_query = text("""
            UPDATE customers 
            SET customer_name = :customer_name,
                gender = :gender,
                birth_date = :birth_date,
                email = :email,
                phone_number = :phone_number,
                document_type = :document_type,
                document_number = :document_number,
                is_identity_verified = :is_identity_verified
            WHERE customer_id = :customer_id
        """)
        
        # Execute in batches of 1000 records
        batch_size = 1000
        for i in range(0, len(updates), batch_size):
            conn.execute(update_query, updates[i:i + batch_size])
            
        print("✅ Customer identities (`customer_name`), contacts, and verification records updated successfully!")

update_customer_identity_and_contact()

Checking schema consistency and fetching customer records...
✅ Customer identities (`customer_name`), contacts, and verification records updated successfully!


### 2.3 Travel Profiles & Platform Preferences (`travel_profile`, `preferred_payment_method`, `preferred_language`, `preferred_currency`)

Categorizing customers by primary travel intent (Leisure, Business, Digital Nomad), establishing default global application settings (`preferred_language`, `preferred_currency`), and assigning preferred transaction mechanisms (`Credit Card`, `PayPal`, `Bank Transfer`).

In [10]:
def update_customer_preferences():
    with engine.begin() as conn:
        print("Fetching customer records to assign behavioral preferences...")
        customers = conn.execute(text("SELECT customer_id FROM customers")).fetchall()
        
        travel_profiles = ['Leisure', 'Business', 'Digital Nomad']
        travel_weights = [0.65, 0.25, 0.10]
        
        payment_methods = ['Credit Card', 'PayPal', 'Bank Transfer', 'Debit Card']
        payment_weights = [0.60, 0.25, 0.10, 0.05]
        
        languages = ['en-US', 'es-ES', 'fr-FR', 'de-DE']
        lang_weights = [0.80, 0.10, 0.05, 0.05]
        
        currencies = ['USD', 'EUR', 'CAD', 'GBP']
        curr_weights = [0.82, 0.10, 0.05, 0.03]
        
        updates = []
        
        for (customer_id,) in customers:
            profile = random.choices(travel_profiles, weights=travel_weights)[0]
            payment = random.choices(payment_methods, weights=payment_weights)[0]
            lang = random.choices(languages, weights=lang_weights)[0]
            curr = random.choices(currencies, weights=curr_weights)[0]
            
            updates.append({
                "customer_id": customer_id,
                "travel_profile": profile,
                "preferred_payment_method": payment,
                "preferred_language": lang,
                "preferred_currency": curr
            })
            
        update_query = text("""
            UPDATE customers 
            SET travel_profile = :travel_profile,
                preferred_payment_method = :preferred_payment_method,
                preferred_language = :preferred_language,
                preferred_currency = :preferred_currency
            WHERE customer_id = :customer_id
        """)
        
        batch_size = 1000
        for i in range(0, len(updates), batch_size):
            conn.execute(update_query, updates[i:i + batch_size])
            
        print("✅ Customer travel profiles and platform preferences updated successfully!")

update_customer_preferences()

Fetching customer records to assign behavioral preferences...
✅ Customer travel profiles and platform preferences updated successfully!


### 2.4 Billing Address & Spatial Metadata (`street`, `city`, `state`, `zip_code`, `country`)

Populating localized billing addresses across the `customers` table using `Faker` to establish customer geographic origin for spatial origin-destination analytics.

In [11]:
def update_customer_billing_addresses():
    with engine.begin() as conn:
        print("Fetching customer records to generate billing addresses...")
        customers = conn.execute(text("SELECT customer_id FROM customers")).fetchall()
        
        updates = []
        
        for (customer_id,) in customers:
            street = fake.street_address()
            city = fake.city()
            state = fake.state_abbr()
            zip_code = fake.zipcode()
            country = 'United States'  # Primary market baseline
            
            updates.append({
                "customer_id": customer_id,
                "street": street,
                "city": city,
                "state": state,
                "zip_code": zip_code,
                "country": country
            })
            
        update_query = text("""
            UPDATE customers 
            SET street = :street,
                city = :city,
                state = :state,
                zip_code = :zip_code,
                country = :country
            WHERE customer_id = :customer_id
        """)
        
        batch_size = 1000
        for i in range(0, len(updates), batch_size):
            conn.execute(update_query, updates[i:i + batch_size])
            
        print("✅ Customer billing addresses updated successfully!")

update_customer_billing_addresses()

Fetching customer records to generate billing addresses...
✅ Customer billing addresses updated successfully!


### 2.5 Temporal Integrity Enforcer (`joined_at`, `last_login_at`)

Generating platform registration timestamps (`joined_at`) and recent activity markers (`last_login_at`) to ensure proper temporal order (registration strictly precedes last login) for cohort retention analysis.

In [12]:
def update_customer_temporal_markers():
    with engine.begin() as conn:
        print("Fetching customer records to generate temporal timestamps...")
        customers = conn.execute(text("SELECT customer_id FROM customers")).fetchall()
        
        # Baseline registration window (2020-01-01 to 2024-12-31)
        base_start = datetime.datetime(2020, 1, 1)
        base_end = datetime.datetime(2024, 12, 31)
        total_seconds_window = int((base_end - base_start).total_seconds())
        
        ref_now = datetime.datetime(2026, 1, 1)
        
        updates = []
        
        for (customer_id,) in customers:
            # 1. Generate random joined_at date
            random_sec_joined = random.randint(0, total_seconds_window)
            joined_at = base_start + datetime.timedelta(seconds=random_sec_joined)
            
            # 2. Generate last_login_at (between joined_at and ref_now)
            remaining_seconds = int((ref_now - joined_at).total_seconds())
            random_sec_login = random.randint(0, remaining_seconds)
            last_login_at = joined_at + datetime.timedelta(seconds=random_sec_login)
            
            updates.append({
                "customer_id": customer_id,
                "joined_at": joined_at.strftime('%Y-%m-%d %H:%M:%S'),
                "last_login_at": last_login_at.strftime('%Y-%m-%d %H:%M:%S')
            })
            
        update_query = text("""
            UPDATE customers 
            SET joined_at = :joined_at,
                last_login_at = :last_login_at
            WHERE customer_id = :customer_id
        """)
        
        batch_size = 1000
        for i in range(0, len(updates), batch_size):
            conn.execute(update_query, updates[i:i + batch_size])
            
        print("✅ Temporal markers (`joined_at`, `last_login_at`) successfully updated!")

update_customer_temporal_markers()

Fetching customer records to generate temporal timestamps...
✅ Temporal markers (`joined_at`, `last_login_at`) successfully updated!


### 2.6 Governance, Privacy (`marketing_opt_in`) & Final Inspection

Assigning promotional communication consent flags (`marketing_opt_in`) to simulate compliance standards (GDPR/LGPD), followed by a full table inspection to validate schema consistency and enriched feature quality.

In [13]:
def update_marketing_opt_in_and_verify():
    with engine.begin() as conn:
        print("Calibrating `marketing_opt_in` flags across customer profiles...")
        customers = conn.execute(text("SELECT customer_id FROM customers")).fetchall()
        
        updates = []
        for (customer_id,) in customers:
            # ~75% opt-in rate
            opt_in = 1 if random.random() < 0.75 else 0
            updates.append({"customer_id": customer_id, "marketing_opt_in": opt_in})
            
        update_query = text("""
            UPDATE customers 
            SET marketing_opt_in = :marketing_opt_in 
            WHERE customer_id = :customer_id
        """)
        
        batch_size = 1000
        for i in range(0, len(updates), batch_size):
            conn.execute(update_query, updates[i:i + batch_size])
            
        print("✅ Privacy flag `marketing_opt_in` successfully initialized!")

update_marketing_opt_in_and_verify()

# Final Data Verification
def verify_customers_refactoring():
    with engine.connect() as conn:
        df = pd.read_sql(text("SELECT * FROM customers LIMIT 10"), conn)
        print("\n🔍 Sample Refactored Customers Data:")
        print(df)

verify_customers_refactoring()

Calibrating `marketing_opt_in` flags across customer profiles...
✅ Privacy flag `marketing_opt_in` successfully initialized!

🔍 Sample Refactored Customers Data:
  customer_id   customer_name customer_tax_id                       email  \
0           1  Cody Robertson  827.613.094-13  cody.robertson@outlook.com   
1          10  Timothy Harper  147.538.296-00    timothy.harper@gmail.com   
2         100   Michael Evans  830.924.165-89     michael.evans@yahoo.com   
3        1000   Katelyn Allen  810.325.746-07   katelyn.allen@outlook.com   
4       10000  Douglas Miller  019.756.284-11    douglas.miller@yahoo.com   
5        1001   Brittany Rowe  078.463.259-65   brittany.rowe@outlook.com   
6        1002  Virginia Jones  509.324.781-23   virginia.jones@icloud.com   
7        1003   Tina Peterson  483.270.695-00   tina.peterson@outlook.com   
8        1004    Tyler Suarez  408.153.796-84      tyler.suarez@yahoo.com   
9        1005   Regina Turner  427.693.508-38   regina.turner@outloo